# 04 - Split creation (frozen, hashed, idempotent)

**Runtime -> Run all. Safe to re-run any number of times.**

Splits are files, not functions. Each is written once, hashed, and consumed by
filename forever after. On re-run, a split whose files already exist is
*verified against its recorded hashes and left alone* - never regenerated.
Regeneration is the failure mode this notebook exists to prevent: a library
upgrade or a changed row order would silently shift membership and make every
number produced before and after incomparable.

Three splits:

* **random** - the conventional split. Optimistic; reported as the baseline.
* **family_disjoint** - whole DGA families held out. The honest number: it
  measures generalisation to families never seen in training, and defeats
  memorisation of family artefacts such as the 12 fixed-length generators.
* **temporal** - train on past, test on future. Only URLhaus rows carry dates,
  so this is small; reported as a limited secondary analysis, and it grows as
  the daily feed snapshots accumulate.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard

In [ ]:
import pandas as pd
from pathlib import Path
from src.evaluate import splits

SPLIT_DIR = Path(P['data']['splits'])
df = pd.read_parquet(f"{P['data']['interim']}/domains_labelled.parquet")
print(df.shape)
print(df['label'].value_counts().to_dict())
print('families:', df[df.label==1]['family'].nunique())

## Create-or-verify

For each split: if its metadata file exists, load and hash-verify it (a
mismatch raises immediately - that means someone touched a frozen file). Only
a split that does not exist yet is created.

In [ ]:
import json

def create_or_verify(name, create_fn):
    meta_file = SPLIT_DIR / f'{name}_metadata.json'
    if meta_file.exists():
        loaded = splits.load_split(SPLIT_DIR, name)     # raises on hash mismatch
        meta = loaded['metadata']
        print(f'{name:22s} EXISTS, hashes verified  n={meta["n"]}')
        return meta
    meta = create_fn()
    print(f'{name:22s} CREATED                  n={meta["n"]}')
    return meta

m_random = create_or_verify('random_v1',
    lambda: splits.random_split(df, SPLIT_DIR, 'random_v1', seed=42))

m_family = create_or_verify('family_disjoint_v1',
    lambda: splits.family_disjoint_split(df, SPLIT_DIR, 'family_disjoint_v1', seed=42))

In [ ]:
# Temporal: only rows with a usable date participate. Guarded because almost
# all dated rows come from URLhaus; if there are too few, the split is skipped
# with a note rather than silently producing a meaningless evaluation.
dated = pd.to_datetime(df['first_seen'], errors='coerce', utc=True).notna().sum()
print('rows with a usable first_seen date:', dated)

if dated >= 150:
    m_temporal = create_or_verify('temporal_v1',
        lambda: splits.temporal_split(df, SPLIT_DIR, 'temporal_v1',
                                      date_col='first_seen'))
else:
    m_temporal = None
    print('SKIPPED temporal_v1: too few dated rows for a meaningful split. '
          'Re-run after daily feed snapshots accumulate.')

## Verification

Three invariants, checked every run:

1. train / val / test are pairwise disjoint;
2. no DGA family appears on both sides of the family-disjoint split;
3. the class balance of each part is sane (no part accidentally single-class).

In [ ]:
def verify(name):
    sp = splits.load_split(SPLIT_DIR, name)
    d = sp['domains']
    assert not (d['train'] & d['test']), f'{name}: train/test overlap'
    assert not (d['train'] & d['val']),  f'{name}: train/val overlap'
    assert not (d['val'] & d['test']),   f'{name}: val/test overlap'
    for part in ('train','val','test'):
        sub = df[df['domain'].isin(d[part])]
        rate = sub['label'].mean()
        print(f'  {name:20s} {part:5s} n={len(sub):>9,}  malicious={rate:.3f}')
    return sp

sp_r = verify('random_v1')
sp_f = verify('family_disjoint_v1')

fams = sp_f['metadata']
tr, te = set(fams['train_families']), set(fams['test_families'])
assert not (tr & te), 'family leak between train and test'
print()
print('train families:', len(tr), '| val:', len(fams['val_families']),
      '| TEST (held out):', len(te))
print('held-out test families:', sorted(te))

In [ ]:
# Where do the 12 fixed-length generators land? Either side is workable, but
# it changes how the family-disjoint generalisation gap should be read.
FIXED = df[df.label==1].groupby('family')['domain'].apply(
    lambda s: s.str.split('.').str[0].str.len().std())
fixed_fams = set(FIXED[FIXED < 0.5].index)
print('fixed-length families:', len(fixed_fams))
print('  in TRAIN:', sorted(fixed_fams & tr))
print('  in TEST :', sorted(fixed_fams & te))

## Record

In [ ]:
summary = {n: json.loads((SPLIT_DIR/f'{n}_metadata.json').read_text())['n']
           for n in ['random_v1','family_disjoint_v1']
           if (SPLIT_DIR/f'{n}_metadata.json').exists()}
if (SPLIT_DIR/'temporal_v1_metadata.json').exists():
    summary['temporal_v1'] = json.loads((SPLIT_DIR/'temporal_v1_metadata.json').read_text())['n']
print(json.dumps(summary, indent=2))
print()
for f in sorted(SPLIT_DIR.iterdir()):
    print(f'{f.stat().st_size/1e6:7.2f} MB  {f.name}')

---

Splits are frozen. From here on, nothing regenerates them; every notebook
consumes them by name and the hash check guards against accidental edits.

**Next:** `03_feature_engineering` - which must load `family_disjoint_v1` to
fit the n-gram model on training benign domains only.